# 文本相似度实例

## Step1 导入相关包

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification,Trainer,TrainingArguments
from datasets import load_dataset

## Step2 加载数据集

In [ ]:
dataset = load_dataset("json", data_files="./train_pair_1w.json", split="train")
dataset

In [ ]:
dataset[0]

## Step3 划分数据集

In [ ]:
datasets = dataset.train_test_split(test_size=0.2)
datasets

## Step4 数据预处理

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

def process_function(examples):
    sentences = [] # 存所有句子
    labels = [] # 存所有标签
    for sentence1, sentence2, label in zip(examples["sentence1"], examples["sentence2"], examples["label"]):
        sentences.append(sentence1) # 每对句子作为一个输入
        sentences.append(sentence2)
        labels.append(1 if label == 1 else -1) # 标签需要是0或1
    tokenized_examples = tokenizer(sentences,max_length=128, truncation=True,padding = "max_length")  # ty:ignore[call-non-callable]
    tokenized_examples = {k:[v[i:i+2] for i in range(0,len(v),2)] for k,v in tokenized_examples.items()}
    tokenized_examples["labels"] = labels
    return tokenized_examples
tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets["train"][0]

In [ ]:
print(tokenized_datasets["train"][0])

## Step5 创建模型

In [ ]:
# 在BertForSequenceClassification源码中看针对特定任务的模型的类都继承自BertPreTrainedModel
# 我们自定义的模型也需要继承BertPreTrainedModel
# 照着源码写模型类，根据实际需求进行修改
from typing import Unpack
from transformers import BertForSequenceClassification, BertPreTrainedModel
from transformers import BertModel
import torch
from transformers.utils import TransformersKwargs
from transformers.modeling_outputs import SequenceClassifierOutput
from torch.nn import CosineSimilarity, CosineEmbeddingLoss


class MyDualModel(BertPreTrainedModel):
    def __init__(self, config):
        # 这里照着BertForSequenceClassification的源码写，调用父类的初始化方法
        # 设置BertModel作为子模块，但是由于本任务只需要向量表示，所以去掉Linear分类头
        super().__init__(config)
        self.config = config
        self.bert = BertModel(config)
        self.post_init()

    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | SequenceClassifierOutput:

        # step1 分别获取sentence1和sentence2的输入
        sen1_input_ids, sen2_input_ids = input_ids[:, 0], input_ids[:, 1]
        sen1_attention_mask, sen2_attention_mask = (
            attention_mask[:, 0],
            attention_mask[:, 1],
        )
        sen1_token_type_ids, sen2_token_type_ids = (
            token_type_ids[:, 0],
            token_type_ids[:, 1],
        )

        # step2 分别通过BertModel获取sentence1和sentence2的输出
        # 对着源码修改
        sen1_outputs = self.bert(
            input_ids=sen1_input_ids,
            attention_mask=sen1_attention_mask,
            token_type_ids=sen1_token_type_ids,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            return_dict=True,
            **kwargs,
        )
        sen1_pooled_output = sen1_outputs[1]  # [batch, hidden_size]

        sen2_outputs = self.bert(
            input_ids=sen2_input_ids,
            attention_mask=sen2_attention_mask,
            token_type_ids=sen2_token_type_ids,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            return_dict=True,
            **kwargs,
        )
        sen2_pooled_output = sen2_outputs[1]  # [batch, hidden_size]

        # step3 计算sentence1和sentence2的余弦相似度
        cos_sim = CosineSimilarity(dim=1)
        cos_sim_score = cos_sim(sen1_pooled_output, sen2_pooled_output)  # [batch]

        # step4 计算loss
        loss = None
        if labels is not None:
            loss_fct = CosineEmbeddingLoss(margin=0.3)
            loss = loss_fct(sen1_pooled_output, sen2_pooled_output, labels)

        # step5 输出结果 如果有labels，就输出loss，并且第一个元素是loss（这是transformers的约定），否则只输出余弦相似度分数
        output = (cos_sim_score,)  # 输出余弦相似度分数
        return ((loss,) + output) if loss is not None else output

model = MyDualModel.from_pretrained("hfl/chinese-macbert-base")


## Step6 创建评估函数

In [ ]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [ ]:
def eval_metric(eval_predict):
    predicts, labels = eval_predict
    # predicts labels 是 (2000, 1)的二维数组，predicts[0]是(1,)的一维数组，无法直接int转换
    predicts = predicts.squeeze()
    labels = labels.squeeze()

    predicts = (predicts > 0.7).astype(int)
    labels = (labels>0).astype(int)

    acc = acc_metric.compute(predictions=predicts, references=labels) or {}  # ty:ignore[missing-argument]
    f1 = f1_metric.compute(predictions=predicts, references=labels) or {}  # ty:ignore[missing-argument]
    acc.update(f1)
    return acc


## Step7 创建TrainingArguments

In [ ]:
train_args = TrainingArguments(
    output_dir="./dual_model",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    metric_for_best_model="f1",
    load_best_model_at_end=True,
)

## Step8 创建Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_datasets["train"].select(range(100)),
    eval_dataset=tokenized_datasets["test"].select(range(100)),
    compute_metrics=eval_metric,
)

## Step9 模型训练

In [ ]:
trainer.train()

## Step10 模型评估


In [ ]:
trainer.evaluate(tokenized_datasets["test"])

## Step11 模型预测

In [ ]:
class SentenceSimilarityPipeline:
    def __init__(self, model, tokenizer):
        self.model = model.bert  # 直接使用BertModel获取句子向量表示
        self.tokenizer = tokenizer
        self.device = model.device

    def preprocess(self, sentence1, sentence2):
        return self.tokenizer(
            [sentence1, sentence2],
            max_length=128,
            truncation=True,
            return_tensors="pt",
            padding=True,
        )

    def predict(self, inputs):
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        return self.model(**inputs)[1]  # 直接返回句子向量表示 [2, hidden_dim=768]

    def postprocess(self, logits):
        cos = (
            CosineSimilarity()(logits[None, 0, :], logits[None, 1, :])
            .squeeze()
            .to("cpu")
            .item()
        )
        return cos

    def __call__(self, sentence1, sentence2, return_vector=False):
        inputs = self.preprocess(sentence1, sentence2)
        predicts = self.predict(inputs)
        results = self.postprocess(predicts)
        if return_vector:
            return results, predicts
        return results


In [ ]:
pipe = SentenceSimilarityPipeline(model, tokenizer)
sentence1 = "今天天气真好"
sentence2 = "今天天气真好"
pipe(sentence1, sentence2, return_vector=True)